In [1]:
from google.colab import drive

# Mount Drive to grab your LoRA
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install -U transformers peft trl bitsandbytes accelerate datasets pandas vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 150.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("🚀 Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 1. Download the config and strip out the hardcoded FP8 rules (The Fix!)
print("⚙️ Patching Model Config...")
config = AutoConfig.from_pretrained(MODEL_ID)
if hasattr(config, "quantization_config"):
    del config.quantization_config
    print("✅ Removed hardcoded FP8 config.")

# 2. Define our explicit 4-bit BitsAndBytes config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Upgraded to A100's native bf16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

print("🔗 Prepping LoRA (Parameter Efficient Fine-Tuning)...")
model = prepare_model_for_kbit_training(model)

# 4. The maximum-capacity LoRA config
lora_config = LoraConfig(
    r=16,          # 🚀 HALVED: Massive speedup for the backward pass
    lora_alpha=32, # Adjusted to match r=16
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

🚀 Loading Tokenizer...
⚙️ Patching Model Config...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔗 Prepping LoRA (Parameter Efficient Fine-Tuning)...
trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


In [ ]:
import pandas as pd
from datasets import Dataset

print("📂 Formatting Kaggle Dataset...")
df = pd.read_csv("train.csv")
formatted_data = []

SYSTEM_PROMPT = (
    "You are an expert SVG coder. Generate valid SVG markup from user requests. "
    "Return ONLY the SVG element with no explanation. "
    "Requirements: xmlns='http://www.w3.org/2000/svg', width='256', height='256', viewBox='0 0 256 256'."
)

for _, row in df.iterrows():
    # Strict ChatML Formatting
    text = (f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{row['prompt']}<|im_end|>\n"
            f"<|im_start|>assistant\n{row['svg']}<|im_end|>")
    formatted_data.append({"text": text})

dataset = Dataset.from_list(formatted_data)
print(f"✅ Prepared {len(dataset)} training examples.")

📂 Formatting Kaggle Dataset...
✅ Prepared 50000 training examples.


In [ ]:

from trl import SFTTrainer, SFTConfig

print("⚡ Starting High-Speed Training...")
sft_config = SFTConfig(
    output_dir="outputs",
    num_train_epochs=4,
    per_device_train_batch_size=4, # 🚀 Doubled physical batch size to saturate the A100
    gradient_accumulation_steps=4, # 4 * 4 = Effective Batch Size of 16
    learning_rate=1e-4,
    warmup_steps=10,
    bf16=True,
    logging_steps=5,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=2048, # 🚀 MASSIVE SPEEDUP: Quadratic reduction in attention compute
    packing=False,   # 🚀 THE FIX: Stops the step count from exploding
)

trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=dataset, args=sft_config)
trainer.train()

print("💾 Saving LoRA Adapter...")
trainer.model.save_pretrained("qwen_2.5_svg_lora")
tokenizer.save_pretrained("qwen_2.5_svg_lora")
print("✅ Native Hugging Face Training Complete!")

⚡ Starting High-Speed Training...


Adding EOS to train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.910000


In [ ]:
from google.colab import drive
import os

# 1. Mount your Google Drive
# A popup will appear asking for permission to connect to your Google account.
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define the exact folder path in your Drive
# This will create a folder named 'Kaggle_SVG_Qwen3.5_LoRA' in your main Drive directory
DRIVE_SAVE_PATH = "/content/drive/MyDrive/Kaggle_SVG_Qwen2.5_LoRA_v2"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# 3. Save the LoRA adapter and tokenizer directly to that folder
print(f"💾 Saving LoRA Adapter directly to: {DRIVE_SAVE_PATH} ...")
trainer.model.save_pretrained(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)

print("✅ Model weights successfully secured in Google Drive!")

In [3]:
import time
import re
import xml.etree.ElementTree as ET
import pandas as pd
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest


LORA_PATH = "/content/drive/MyDrive/Kaggle_SVG_Qwen2.5_LoRA_v2"

print("🚀 Booting up vLLM Engine in Colab-Safe Mode...")
llm = LLM(
    model="Qwen/Qwen2.5-Coder-3B-Instruct",
    enable_lora=True,
    max_lora_rank=32,
    gpu_memory_utilization=0.85, # Leaves 15% VRAM free to prevent crashes
    max_model_len=6144,          # Massive 6k token breathing room!
    dtype="bfloat16",
    enforce_eager=True           # THE FIX: Bypasses Colab's 64MB Shared Memory limit
)

print("⚙️ Setting up Best-of-5 Sampling...")
sampling_params = SamplingParams(
    n=5,
    temperature=0.6,
    max_tokens=6144,
    stop=["<|im_end|>"]
)

print("📂 Preparing Kaggle Prompts...")
test_df = pd.read_csv("test.csv")
prompts = test_df["prompt"].tolist()
ids = test_df["id"].tolist()

SYSTEM_PROMPT = (
    "You are an expert SVG coder. Generate valid SVG markup from user requests. "
    "Return ONLY the SVG element with no explanation. "
    "Requirements: xmlns='http://www.w3.org/2000/svg', width='256', height='256', viewBox='0 0 256 256'."
)

formatted_prompts = [
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n"
    for p in prompts
]

print(f"⚡ Launching vLLM Generation for 1000 prompts...")
t0 = time.time()

# BOOM. All generation happens in this single line.
outputs = llm.generate(
    formatted_prompts,
    sampling_params,
    lora_request=LoRARequest("svg_adapter", 1, LORA_PATH)
)

print("🔍 Validating XML Outputs...")
ET.register_namespace("", "http://www.w3.org/2000/svg")
SVG_FULL_RE = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)
fallback_svg = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="black"/></svg>'

def extract_and_validate(text):
    if not text.strip().endswith("</svg>"): text += "</svg>"
    m = SVG_FULL_RE.search(text)
    svg = m.group(0).strip() if m else ""
    try:
        if svg and ET.fromstring(svg).tag.endswith("svg"): return svg
    except: pass
    return None

rows = []
fallbacks_used = 0

for i, output in enumerate(outputs):
    valid_svg = None
    for attempt in output.outputs:
        valid_svg = extract_and_validate(attempt.text)
        if valid_svg:
            break

    if not valid_svg:
        fallbacks_used += 1
        valid_svg = fallback_svg

    rows.append({"id": ids[i], "svg": valid_svg})

pd.DataFrame(rows).to_csv("submission_vllm_final.csv", index=False)
print(f"\n🎉 DONE! Generated and validated in {(time.time()-t0)/60:.2f} mins.")
print(f"Total Fallbacks: {fallbacks_used}/{len(prompts)}")

🚀 Booting up vLLM Engine in Colab-Safe Mode...
INFO 04-02 01:38:36 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 6144, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'max_lora_rank': 32, 'model': 'Qwen/Qwen2.5-Coder-3B-Instruct'}
INFO 04-02 01:38:37 [model.py:533] Resolved architecture: Qwen2ForCausalLM
INFO 04-02 01:38:37 [model.py:1582] Using max model len 6144
INFO 04-02 01:38:37 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-02 01:38:37 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 04-02 01:38:38 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-02 01:38:38 [vllm.py:820] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-02 01:38:38 [vllm.py:985] Cudagr

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [3]:
!pip install --upgrade transformers peft accelerate

  Using cached transformers-5.4.0-py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 51.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.18.1 requires transformers<5,>=4.56.0, but you have transformers 5.4.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


In [5]:
import torch
import pandas as pd
import re
import time
import xml.etree.ElementTree as ET
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
LORA_PATH = "/content/drive/MyDrive/Kaggle_SVG_Qwen2.5_LoRA"
TEST_CSV = "test.csv"
OUTPUT_HTML = "ablation_matrix.html"

MAX_TOKENS = 4096
FALLBACK_SVG = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect width="256" height="256" fill="#ffebee"/><text x="128" y="128" text-anchor="middle" fill="red">FAILED</text></svg>'
SVG_FULL_RE = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)

# Define our 4 Ablation Studies
ABLATIONS = {
    "1_Baseline_Greedy": {"do_sample": False, "temperature": None, "top_p": None, "repetition_penalty": 1.0, "num_return_sequences": 1},
    "2_High_Temp": {"do_sample": True, "temperature": 0.8, "top_p": 0.95, "repetition_penalty": 1.0, "num_return_sequences": 1},
    "3_Rep_Penalty": {"do_sample": True, "temperature": 0.35, "top_p": 0.90, "repetition_penalty": 1.2, "num_return_sequences": 1},
    "4_The_Champion": {"do_sample": True, "temperature": 0.35, "top_p": 0.90, "repetition_penalty": 1.1, "num_return_sequences": 5} # Best-of-5
}

SYSTEM_PROMPT = (
    "You are an expert SVG coder. Generate valid SVG markup from user requests. "
    "Return ONLY the SVG element with no explanation. "
    "Requirements: xmlns='http://www.w3.org/2000/svg', width='256', height='256', viewBox='0 0 256 256'."
)
print("🚀 Loading Model for Ablation Study...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# THE FIX: Force the model onto GPU 0 instead of letting "auto" guess
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": 0}
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

# (The rest of your code stays exactly the same)
test_df = pd.read_csv(TEST_CSV).head(5) # ONLY THE FIRST 5 PROMPTS

# ==========================================
# ⚡ RUN ABLATIONS
# ==========================================
results = []

def extract_and_validate(text):
    if not text.strip().endswith("</svg>"): text += "</svg>"
    m = SVG_FULL_RE.search(text)
    svg = m.group(0).strip() if m else ""
    try:
        if svg and ET.fromstring(svg).tag.endswith("svg"): return svg
    except: pass
    return None

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing Prompts"):
    item_id = row['id']
    prompt = row['prompt']
    formatted_prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    prompt_results = {"id": item_id, "prompt": prompt}

    for config_name, params in ABLATIONS.items():
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                **params
            )

        candidates_text = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

        valid_candidates = []
        for text in candidates_text:
            valid_svg = extract_and_validate(text)
            if valid_svg: valid_candidates.append(valid_svg)

        if valid_candidates:
            # Shortest valid SVG logic
            best_svg = min(valid_candidates, key=len)
        else:
            best_svg = FALLBACK_SVG

        prompt_results[config_name] = best_svg

    results.append(prompt_results)

# ==========================================
# 🎨 BUILD HTML MATRIX
# ==========================================
print("\n🎨 Building HTML Matrix...")
html = '''
<!DOCTYPE html>
<html>
<head>
    <style>
        body { font-family: system-ui; background: #f4f7f6; padding: 20px; }
        table { border-collapse: collapse; width: 100%; background: white; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        th, td { border: 1px solid #ddd; padding: 15px; text-align: center; vertical-align: top; }
        th { background: #1a1a1a; color: white; position: sticky; top: 0; }
        .prompt-col { text-align: left; width: 250px; font-size: 14px; background: #fafafa; }
        .svg-container { width: 200px; height: 200px; border: 1px dashed #ccc; margin: 0 auto; display: flex; align-items: center; justify-content: center; background: white; overflow: hidden; }
        .svg-container svg { width: 100%; height: 100%; }
        .config-desc { font-size: 12px; color: #888; font-weight: normal; margin-top: 5px; }
    </style>
</head>
<body>
    <h2>🧪 SVG Generation Ablation Study</h2>
    <table>
        <tr>
            <th>Prompt</th>
            <th>1. Baseline (Greedy)<div class="config-desc">Temp: 0.0 | No Penalty</div></th>
            <th>2. High Temp<div class="config-desc">Temp: 0.8 | Creative Risk</div></th>
            <th>3. Repetition Penalty<div class="config-desc">Temp: 0.35 | Rep: 1.2</div></th>
            <th>4. The Champion<div class="config-desc">Temp: 0.35 | Rep: 1.1 | Best-of-5</div></th>
        </tr>
'''

for r in results:
    html += f'''
        <tr>
            <td class="prompt-col"><strong>{r['id'][:8]}...</strong><br><br>{r['prompt']}</td>
            <td><div class="svg-container">{r['1_Baseline_Greedy']}</div></td>
            <td><div class="svg-container">{r['2_High_Temp']}</div></td>
            <td><div class="svg-container">{r['3_Rep_Penalty']}</div></td>
            <td style="background: #e8fbed;"><div class="svg-container">{r['4_The_Champion']}</div></td>
        </tr>
    '''

html += "</table></body></html>"

with open(OUTPUT_HTML, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"✅ Ablation study complete! Open {OUTPUT_HTML} to view the matrix.")

🚀 Loading Model for Ablation Study...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Processing Prompts: 100%|██████████| 5/5 [1:28:51<00:00, 1066.26s/it]


🎨 Building HTML Matrix...
✅ Ablation study complete! Open ablation_matrix.html to view the matrix.
